# Random Forest Alpine Terrain Classifier: Tetons First Pass

This notebook is a beginner-friendly, editable workflow for classifying alpine terrain around Grand Teton National Park with Google Earth Engine (GEE), Sentinel imagery, terrain variables, weak labels from Dynamic World, and a Random Forest classifier.

The design borrows a few ideas from Yang et al. (2023), **A novel alpine land cover classification strategy based on a deep convolutional neural network and multi-source remote sensing data in Google Earth Engine**: use multi-source remote sensing data, work in GEE for scalable processing, use existing land-cover products to bootstrap training samples, and inspect terrain/elevation patterns after classification. This notebook intentionally uses Random Forest instead of their DCNN so the first model is easier to understand and modify.

Sources to keep open while you work:

- Paper: https://www.tandfonline.com/doi/full/10.1080/15481603.2023.2233756
- Earth Engine authentication: https://developers.google.com/earth-engine/guides/auth
- Earth Engine supervised classification: https://developers.google.com/earth-engine/guides/classification
- Sentinel-2 SR Harmonized: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED
- Dynamic World V1: https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_DYNAMICWORLD_V1


## 0. Mental Model

A supervised land-cover classifier needs three things:

1. **Predictors**: input bands the model can learn from, such as Sentinel-2 reflectance, vegetation/snow/water indices, Sentinel-1 radar, elevation, slope, and aspect.
2. **Labels**: examples of known classes. For this first pass, we use high-confidence Dynamic World labels as weak labels. Later, you can replace or supplement these with hand-digitized training polygons.
3. **Evaluation**: a train/test split and confusion matrix, so you can see where the model is strong or confused.

The first model is not meant to be publishable. It is meant to make the workflow legible.


## 1. Imports and Earth Engine Login

Run this once. If Earth Engine asks for a Cloud project, use the project connected to your Earth Engine account. If you do not know it yet, leave `EE_PROJECT` as `None`, run the cell, and follow the error message or authentication prompt.


In [ ]:
import os
import ee
import geemap
import pandas as pd

EE_PROJECT = None  # Example: "my-earth-engine-project"

try:
    if EE_PROJECT:
        ee.Initialize(project=EE_PROJECT)
    else:
        ee.Initialize()
    print("Earth Engine initialized.")
except Exception as exc:
    print("Authentication needed or initialization failed:", exc)
    ee.Authenticate()
    if EE_PROJECT:
        ee.Initialize(project=EE_PROJECT)
    else:
        ee.Initialize()
    print("Earth Engine initialized after authentication.")


## 2. Study Area

This starter area covers the Grand Teton alpine zone. Keep it small while learning so maps render quickly and sampling/export tasks do not become expensive.

You can change the rectangle coordinates or draw/import a more precise boundary later.


In [ ]:
# Rectangle roughly around Grand Teton National Park and adjacent alpine terrain.
aoi = ee.Geometry.Rectangle([-111.05, 43.35, -110.45, 44.05])

# Alpine-ish mask threshold for this first pass. Adjust after inspecting the map.
ALPINE_MIN_ELEV_M = 2200
YEAR = 2023
START_DATE = f"{YEAR}-06-15"
END_DATE = f"{YEAR}-09-20"
SCALE = 10
SEED = 42

Map = geemap.Map(center=[43.75, -110.75], zoom=10)
Map.addLayer(aoi, {"color": "yellow"}, "AOI")
Map


## 3. Build Predictor Bands

This section creates a single multi-band image for classification:

- Sentinel-2 summer median surface reflectance
- Spectral indices: NDVI, NDWI, NDSI, BSI
- Sentinel-1 summer median VV/VH radar backscatter
- NASADEM elevation, slope, and aspect

The paper uses multi-source data because alpine classes can look similar in one sensor alone. Snow, bare rock, wet meadows, shrublands, and forest edges are easier to separate when optical, radar, and terrain information work together.


In [ ]:
def mask_s2_sr(image):
    """Mask clouds/cirrus using the Sentinel-2 QA60 bit flags, then scale reflectance."""
    qa = image.select("QA60")
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(cirrus_bit).eq(0))
    return image.updateMask(mask).divide(10000).copyProperties(image, ["system:time_start"])


def add_spectral_indices(image):
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndwi = image.normalizedDifference(["B3", "B8"]).rename("NDWI")
    ndsi = image.normalizedDifference(["B3", "B11"]).rename("NDSI")
    bsi = image.expression(
        "((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))",
        {
            "SWIR": image.select("B11"),
            "RED": image.select("B4"),
            "NIR": image.select("B8"),
            "BLUE": image.select("B2"),
        },
    ).rename("BSI")
    return image.addBands([ndvi, ndwi, ndsi, bsi])

s2_bands = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]

s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
    .map(mask_s2_sr)
    .select(s2_bands)
)

s2_composite = add_spectral_indices(s2.median()).clip(aoi)

s1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    .select(["VV", "VH"])
    .median()
    .clip(aoi)
)

terrain = ee.Algorithms.Terrain(ee.Image("NASA/NASADEM_HGT/001").select("elevation")).select(
    ["elevation", "slope", "aspect"]
).clip(aoi)

alpine_mask = terrain.select("elevation").gte(ALPINE_MIN_ELEV_M)

predictor_image = s2_composite.addBands(s1).addBands(terrain).updateMask(alpine_mask)
predictor_bands = predictor_image.bandNames()

print("Predictor bands:", predictor_bands.getInfo())


## 4. Create Weak Labels from Dynamic World

Dynamic World is a global 10 m land-use/land-cover product with class probabilities. Here, we use it to generate training points only where its class confidence is high.

This follows the spirit of the paper's automated training-sample idea, but keep the caveat in mind: weak labels can encode mistakes from the source product. Later, you should compare against hand-made validation points in places you know well.


In [ ]:
DW_CLASSES = {
    0: "water",
    1: "trees",
    2: "grass",
    3: "flooded_vegetation",
    4: "crops",
    5: "shrub_and_scrub",
    6: "built",
    7: "bare",
    8: "snow_and_ice",
}

# Alpine-focused subset. You can add/remove classes once you inspect the region.
KEEP_CLASSES = [0, 1, 2, 5, 7, 8]
MIN_DW_CONFIDENCE = 0.65
POINTS_PER_CLASS = 250

probability_bands = [
    "water", "trees", "grass", "flooded_vegetation", "crops",
    "shrub_and_scrub", "built", "bare", "snow_and_ice",
]

dw_collection = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .select(["label"] + probability_bands)
)

# Dynamic World label is a class index. Use mode for labels, and median probabilities for confidence.
dw_label = dw_collection.select("label").mode().toInt().rename("class").clip(aoi)
dw_probabilities = dw_collection.select(probability_bands).median().clip(aoi)
dw_confidence = dw_probabilities.reduce(ee.Reducer.max()).rename("dw_confidence")
class_mask = dw_label.remap(KEEP_CLASSES, [1] * len(KEEP_CLASSES), 0).eq(1)
label_image = dw_label.updateMask(class_mask).updateMask(dw_confidence.gte(MIN_DW_CONFIDENCE)).updateMask(alpine_mask)

sampling_image = predictor_image.addBands(label_image)

samples = sampling_image.stratifiedSample(
    numPoints=POINTS_PER_CLASS,
    classBand="class",
    region=aoi,
    scale=SCALE,
    seed=SEED,
    geometries=True,
    tileScale=4,
)

print("Sample count:", samples.size().getInfo())
print("Class histogram:", samples.aggregate_histogram("class").getInfo())


## 5. Train/Test Split and Random Forest

Earth Engine's Random Forest classifier runs server-side. The split below keeps 70% of samples for training and 30% for validation.


In [ ]:
samples = samples.randomColumn("split", seed=SEED)
training = samples.filter(ee.Filter.lt("split", 0.7))
validation = samples.filter(ee.Filter.gte("split", 0.7))

rf = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    variablesPerSplit=None,
    minLeafPopulation=1,
    bagFraction=0.632,
    seed=SEED,
).train(
    features=training,
    classProperty="class",
    inputProperties=predictor_bands,
)

classified = predictor_image.classify(rf).rename("classification")

train_matrix = rf.confusionMatrix()
validated = validation.classify(rf)
validation_matrix = validated.errorMatrix("class", "classification")

print("Training confusion matrix:")
print(train_matrix.getInfo())
print("Training overall accuracy:", train_matrix.accuracy().getInfo())
print("Validation confusion matrix:")
print(validation_matrix.getInfo())
print("Validation overall accuracy:", validation_matrix.accuracy().getInfo())
print("Validation kappa:", validation_matrix.kappa().getInfo())


## 6. Map the Inputs and Classification

Use this map to sanity-check the result. Look for obvious problems:

- Are permanent snowfields mapped as snow/ice?
- Are high bare ridges separated from meadows?
- Do forests stop near plausible treeline elevations?
- Are water bodies being confused with snow/shadow?


In [ ]:
class_palette = [
    "419bdf",  # water
    "397d49",  # trees
    "88b053",  # grass
    "7a87c6",  # flooded vegetation, unused in first pass
    "e49635",  # crops, unused in first pass
    "dfc35a",  # shrub/scrub
    "c4281b",  # built, unused in first pass
    "a59b8f",  # bare
    "b39fe1",  # snow/ice
]

Map = geemap.Map(center=[43.75, -110.75], zoom=10)
Map.addLayer(aoi, {"color": "yellow"}, "AOI")
Map.addLayer(s2_composite, {"bands": ["B4", "B3", "B2"], "min": 0.02, "max": 0.3}, "Sentinel-2 true color")
Map.addLayer(terrain.select("elevation"), {"min": 1800, "max": 4200, "palette": ["2166ac", "f7f7f7", "b2182b"]}, "Elevation")
Map.addLayer(label_image, {"min": 0, "max": 8, "palette": class_palette}, "Dynamic World weak labels")
Map.addLayer(classified, {"min": 0, "max": 8, "palette": class_palette}, "Random Forest classification")
Map.addLayer(training, {"color": "white"}, "Training points", False)
Map.addLayer(validation, {"color": "black"}, "Validation points", False)
Map.addLayerControl()
Map


## 7. Variable Importance

Random Forest can report which predictor bands were most useful. Treat this as a clue, not a final scientific claim. Correlated bands and weak labels can distort importance.


In [ ]:
explanation = rf.explain()
importance = ee.Dictionary(explanation.get("importance"))
importance_df = pd.DataFrame(
    importance.getInfo().items(),
    columns=["band", "importance"],
).sort_values("importance", ascending=False)
importance_df


## 8. Area by Class and Elevation Zone

The paper analyzes alpine land cover along elevation gradients. This starter table summarizes class area by broad elevation zones. Change the breakpoints for your local ecological question.


In [ ]:
# Elevation zones in meters. Adjust these after inspecting local terrain.
elev = terrain.select("elevation")
elev_zone = (
    elev.where(elev.lt(2400), 0)
    .where(elev.gte(2400).And(elev.lt(2800)), 1)
    .where(elev.gte(2800).And(elev.lt(3200)), 2)
    .where(elev.gte(3200), 3)
    .rename("elev_zone")
    .updateMask(alpine_mask)
)

area_image = ee.Image.pixelArea().divide(1e6).rename("area_km2").addBands(classified).addBands(elev_zone)

area_table = area_image.reduceRegion(
    reducer=ee.Reducer.sum().group(
        groupField=1,
        groupName="class",
    ),
    geometry=aoi,
    scale=SCALE,
    maxPixels=1e10,
    tileScale=4,
)

print(area_table.getInfo())

# More advanced: grouped by class and elevation zone. Uncomment once the first table works.
# grouped = area_image.reduceRegion(
#     reducer=ee.Reducer.sum().group(groupField=2, groupName="elev_zone").group(groupField=1, groupName="class"),
#     geometry=aoi,
#     scale=SCALE,
#     maxPixels=1e10,
#     tileScale=4,
# )
# grouped.getInfo()


## 9. Export Options

Keep exports commented until you like the map. Earth Engine exports run as Tasks; you may need to start them from the Earth Engine Tasks tab or via Python task controls.


In [ ]:
# Export to Google Drive as a GeoTIFF.
# task = ee.batch.Export.image.toDrive(
#     image=classified.toByte(),
#     description=f"tetons_alpine_rf_{YEAR}",
#     folder="gee_exports",
#     fileNamePrefix=f"tetons_alpine_rf_{YEAR}",
#     region=aoi,
#     scale=SCALE,
#     maxPixels=1e10,
# )
# task.start()
# print("Started export task:", task.id)


## 10. Next Iterations

Good next experiments:

1. Replace weak labels with hand-made training polygons for places you know well.
2. Compare Sentinel-2 only vs. Sentinel-2 + Sentinel-1 + terrain.
3. Test different seasons: early summer snow, peak summer vegetation, late summer dryness.
4. Tune Random Forest parameters: number of trees, points per class, confidence threshold.
5. Expand from a rectangle to a precise alpine/ecoregion boundary.
6. Export the classification and inspect it in QGIS.

Record each change in `PROJECT_LOG.md` so the project becomes a research diary instead of a pile of code cells.
